# Kaggle ASR v1 Workflow
Single-run workflow: setup, full v1 pipeline, and artifact export.

## Pre-run in Kaggle UI
- Settings -> Accelerator: GPU (T4)
- Settings -> Internet: ON
- Add Input datasets: code dataset + raw dataset.
- Run Save Version -> Save & Run All (Commit) to persist v1 outputs.

In [ ]:
import glob
import os
import zipfile
from pathlib import Path

if Path('kaggle/requirements_kaggle.txt').exists():
    print('Code already present in current directory')
else:
    candidates = glob.glob('/kaggle/input/**/speech_recognation_code_for_kaggle.zip', recursive=True)
    if not candidates:
        raise FileNotFoundError('Attach code dataset with speech_recognation_code_for_kaggle.zip')
    zip_path = candidates[0]
    target_dir = Path('/kaggle/working/speech_recognation')
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(target_dir)
    os.chdir(target_dir)

assert Path('kaggle/requirements_kaggle.txt').exists(), 'Bootstrap failed'
print('CWD:', Path.cwd())

In [ ]:
!python -m pip install -U pip
!python -m pip install -r kaggle/requirements_kaggle.txt
!python kaggle/prepare_environment.py

In [ ]:
from pathlib import Path

RAW_DIR = '/kaggle/input/YOUR_RAW_DATASET/data/raw'
WORK_DIR_V1 = '/kaggle/working/asr_full_run_v1'

EPOCHS_V1 = 8
BATCH_SIZE_V1 = 8
LR_V1 = 2e-6
HOLDOUT_RATIO = 0.15

assert Path(RAW_DIR).exists(), f'RAW_DIR ne postoji: {RAW_DIR}'
print('RAW_DIR:', RAW_DIR)

In [ ]:
import subprocess

cmd = [
    'python', 'kaggle/run_full_pipeline.py',
    '--raw_dir', RAW_DIR,
    '--work_dir', WORK_DIR_V1,
    '--epochs', str(EPOCHS_V1),
    '--batch_size', str(BATCH_SIZE_V1),
    '--learning_rate', str(LR_V1),
    '--holdout_ratio', str(HOLDOUT_RATIO),
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

summary_v1 = Path(WORK_DIR_V1) / 'metrics' / 'comparison_summary.json'
print(json.dumps(json.loads(summary_v1.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
!python kaggle/export_artifacts.py --source_dir /kaggle/working/asr_full_run_v1 --zip_prefix asr_full_run_v1